# Mini-Projeto 3 — Ontologia de Triagem Médica e Doenças (RDF, RDFS, OWL e SPARQL)

Este repositório contém uma base de conhecimento modelada como grafo de conhecimento (Web Semântica) no domínio médico clínico. O sistema simula um motor de inferência e triagem diagnóstica, inspirado no MYCIN, capaz de cruzar sintomas, doenças, agentes patogênicos, tratamentos e pacientes.

O projeto foi desenvolvido em Python com as bibliotecas `rdflib` e `owlrl`, e implementa consultas programáticas (`g.triples()`), consultas e atualizações declarativas (SPARQL) e uma verificação das construções OWL utilizadas.

> Os valores clínicos da base (letalidade, dosagem, incubação) são ilustrativos, para fins acadêmicos, e não substituem fontes médicas.

---

## 1. Resumo da base e inferência (reasoner OWL-RL)

A base (`ontologia_doencas.ttl`) modela doenças, sintomas, agentes patogênicos, tratamentos, médicos e pacientes. Ao carregar o arquivo e aplicar o reasoner `DeductiveClosure(OWLRL_Semantics)`, o script imprime as seguintes métricas:

| Métrica | Valor |
|---|---|
| Triplas explícitas (carregadas do `.ttl`) | **325** |
| Triplas inferidas pelo reasoner | **692** |
| Total após o fechamento dedutivo | **1017** |
| Inconsistências detectadas na base | **0** |
| Classes | **16** |
| Propriedades | **16** (9 de objeto e 7 de dados) |
| Instâncias (A-Box) | **40** |

As 692 triplas inferidas incluem inferências do domínio (inversas, transitividade, simetria, equivalência de classes, herança de classes, domain e range) e também axiomas padrão do fechamento OWL-RL, como `rdf:type owl:Thing`, `rdfs:Resource` e `owl:sameAs` reflexivo.

---

## 2. Arquivos do projeto

- `ontologia_doencas.ttl`: base de conhecimento em Turtle, com a T-Box (esquema e regras) e a A-Box (instâncias).
- `consultas_doencas.py`: script principal. Carrega a base, aplica o reasoner e executa 17 consultas e atualizações (8 com `g.triples()` e 9 com SPARQL), seguidas de 3 verificações das construções OWL.
- `README.md`: este arquivo.

---

## 3. Estrutura da ontologia (T-Box)

A taxonomia tem 16 classes, organizadas por `rdfs:subClassOf` em duas árvores, com até 3 níveis abaixo da raiz `:EntidadeSaude`.

```text
:EntidadeSaude
 ├── :Doenca
 │    ├── :DoencaInfecciosa
 │    │    ├── :DoencaViral
 │    │    └── :DoencaBacteriana
 │    └── :DoencaCronica
 ├── :Sintoma (owl:equivalentClass :ManifestacaoClinica)
 ├── :ManifestacaoClinica
 ├── :Tratamento
 │    ├── :Medicamento
 │    └── :TerapiaSuporte
 └── :AgentePatogenico

:Pessoa
 ├── :Paciente
 │    └── :PacienteTriagemCritica
 └── :Medico
```

### Propriedades

- **9 propriedades de objeto:** `:apresentaSintoma`, `:sintomaDe`, `:tratadaCom`, `:indicadoPara`, `:causadaPor`, `:evoluiPara`, `:interageCom`, `:diagnosticadoCom`, `:atendidoPor`.
- **7 propriedades de dados:** `:codigoCID`, `:taxaLetalidade`, `:periodoIncubacaoDias`, `:gravidadeSintoma`, `:dosagemMg`, `:idadePaciente`, `:statusClinico`.

Todas têm `rdfs:domain` e `rdfs:range` declarados.

### Construtores OWL utilizados

1. **Propriedades inversas (`owl:inverseOf`):** `:apresentaSintoma` ↔ `:sintomaDe` e `:tratadaCom` ↔ `:indicadoPara`. Declaramos só um sentido, e o reasoner materializa o outro (consultas 1.3 e 1.5 e atualização 2.8).
2. **Propriedade transitiva (`owl:TransitiveProperty`):** `:evoluiPara` (Dengue Clássica → Dengue com Sinais de Alarme → Dengue Grave). O reasoner deduz que a Dengue Clássica também evolui para a Dengue Grave (consulta 1.4).
3. **Propriedade simétrica (`owl:SymmetricProperty`):** `:interageCom`. Declaramos `:losartana :interageCom :ibuprofeno`, e o reasoner infere o sentido oposto; o ASK 2.5 consulta justamente o sentido não declarado.
4. **Equivalência de classes (`owl:equivalentClass`):** `:Sintoma` ≡ `:ManifestacaoClinica`. `:mialgia` e `:anosmia` são declaradas apenas como `:ManifestacaoClinica`; o reasoner as classifica também como `:Sintoma` (consulta 1.7 e verificação 3.3). A `:anosmia` não é ligada a nenhuma doença, então a equivalência é a única causa dessa classificação.
5. **Propriedades funcionais (`owl:FunctionalProperty`):** `:causadaPor` (cada doença infecciosa tem um agente patogênico primário), `:codigoCID` (um código por doença) e `:idadePaciente` (uma idade por paciente). Para a propriedade de objeto `:causadaPor`, ligar duas URIs de agentes à mesma doença faz o reasoner inferir que são o mesmo indivíduo (`owl:sameAs`), como mostra a verificação 3.1. Nas propriedades de dados, a unicidade é uma declaração de modelagem: o reasoner OWL-RL não sinaliza erro quando valores literais distintos são atribuídos, então ela deve ser garantida na entrada dos dados.
6. **Disjunção de classes (`owl:disjointWith`):** entre `:DoencaViral` e `:DoencaBacteriana`, `:DoencaInfecciosa` e `:DoencaCronica`, `:Medicamento` e `:TerapiaSuporte`, e entre `:Doenca` e `:Tratamento`/`:Sintoma`. Um indivíduo classificado em duas classes disjuntas é sinalizado como inconsistente (verificação 3.2).

> Em OWL, `FunctionalProperty` e `disjointWith` não impedem a inserção de dados, como uma restrição de banco relacional faria (isso seria papel do SHACL). Eles fazem o reasoner **inferir** ou **detectar** consequências lógicas.

---

## 4. Base de dados (A-Box — 40 instâncias)

- **6 agentes patogênicos:** vírus da Dengue, Influenza A (H1N1), SARS-CoV-2, bacilo de Koch, meningococo e pneumococo.
- **10 doenças:**
  - 5 virais: Dengue Clássica, Dengue com Sinais de Alarme, Dengue Grave, Influenza A e COVID-19.
  - 3 bacterianas: Tuberculose, Pneumonia Bacteriana e Meningite Meningocócica.
  - 2 crônicas: Hipertensão e Asma Grave.
- **8 sintomas e manifestações clínicas:** Febre Alta, Cefaleia, Mialgia, Anosmia, Dispneia, Tosse Seca, Hemorragia e Rigidez na Nuca.
- **10 tratamentos:**
  - 8 medicamentos: Paracetamol, Dipirona, Oseltamivir, Amoxicilina, Rifampicina, Ceftriaxona, Losartana e Ibuprofeno.
  - 2 terapias de suporte: Hidratação Venosa e Oxigenoterapia.
- **6 pessoas:** 2 médicos e 4 pacientes, com idades e quadros clínicos diferentes.

Observação sobre o CID: a Dengue Clássica e a Dengue com Sinais de Alarme usam `A90` (no SUS, `A91` é reservado à forma hemorrágica/grave, usada na Dengue Grave).

---

## 5. Consultas e operações implementadas

O script `consultas_doencas.py` executa as consultas sobre o grafo expandido pelo reasoner. Cada consulta imprime seu número, seu propósito e, na Parte 1, o resultado esperado. A saída é ordenada, portanto idêntica a cada execução.

### Parte 1: navegação com `g.triples()` (8 consultas)

1. **`(sujeito, predicado, None)`:** sintomas da Dengue Clássica.
2. **`(None, predicado, objeto)`:** doenças que apresentam Dispneia.
3. **`(sujeito, predicado inferido, None)`:** de quais doenças a Febre Alta é sintoma, usando `:sintomaDe` (inferido via `owl:inverseOf`).
4. **`(None, predicado transitivo, None)`:** cadeias de evolução clínica com `:evoluiPara` (inclui o salto inferido por transitividade).
5. **`(sujeito, None, None)`:** todas as afirmações sobre a Rifampicina, incluindo a interação simétrica e a indicação inversa.
6. **`(None, rdf:type, :Doenca)`:** todas as doenças, independentemente do nível da hierarquia (inferência por `rdfs:subClassOf`).
7. **`(None, rdf:type, :Sintoma)`:** os 8 sintomas, incluindo Mialgia e Anosmia, que só são `:Sintoma` por `owl:equivalentClass`.
8. **`(None, None, objeto)`:** tudo que aponta para o Vírus da Dengue e por qual propriedade.

### Parte 2: consultas e atualizações SPARQL (9 operações)

1. **`SELECT` + `FILTER` + `ORDER BY`:** doenças com letalidade acima de 2,0%, da mais letal para a menos letal.
2. **`SELECT` + `VALUES` + `COUNT` + `GROUP BY` + `HAVING` (triagem estilo MYCIN):** dado um paciente com Febre Alta, Cefaleia e Mialgia, ranqueia as doenças pelo número de sintomas compatíveis, depois pela cobertura do quadro da doença e, em empate, pela maior letalidade.
3. **`SELECT` + `OPTIONAL` + `BIND`:** lista doenças e seus agentes patogênicos (opcionais, já que doenças crônicas não têm) e calcula o nível de risco com `IF`.
4. **`SELECT` com múltiplos joins e `GROUP_CONCAT`:** prontuário integrado com paciente, idade, doença, médico e tratamentos.
5. **`ASK`:** verifica se existe interação medicamentosa entre Ibuprofeno e Losartana.
6. **`CONSTRUCT`:** deriva um novo grafo ligando cada paciente diretamente aos tratamentos da sua doença (`:prescricaoSugerida`).
7. **`UPDATE` (`INSERT...WHERE`):** classifica como `:PacienteTriagemCritica` os pacientes com doença de letalidade ≥ 5,0% (mostra antes e depois).
8. **`UPDATE` (`DELETE DATA`):** revisão de protocolo clínico, removendo a indicação do Paracetamol para a Dengue Clássica. Remove 2 triplas: a original e a inversa já materializada pelo reasoner (mostra antes e depois).
9. **`UPDATE` (`DELETE...INSERT...WHERE`):** transição de estado, mudando o `:statusClinico` de "Em Triagem" para "Em Tratamento Supervisionado" nos pacientes que já têm médico atribuído (mostra antes e depois).

### Parte 3: verificação das construções OWL (3 testes)

Cada teste usa uma cópia nova do `.ttl` com uma tripla extra, sem alterar o grafo principal:

1. **`owl:FunctionalProperty`:** liga a Dengue Clássica a um segundo agente em `:causadaPor`; o reasoner infere `owl:sameAs` entre os dois (esperado: `True`).
2. **`owl:disjointWith`:** classifica um indivíduo como `:DoencaViral` e `:DoencaBacteriana` ao mesmo tempo (esperado: 0 inconsistências na base original e 1 ou mais no grafo de teste).
3. **`owl:equivalentClass`:** confirma que `:anosmia` virou `:Sintoma` sem passar por `:apresentaSintoma` (esperado: `True` e `True`).

---

## 6. Como executar

### Pré-requisitos

É necessário ter o Python 3 instalado. Instale as bibliotecas:

```bash
pip install rdflib owlrl
```

### Execução

Com `ontologia_doencas.ttl` e `consultas_doencas.py` na mesma pasta, execute:

```bash
python consultas_doencas.py
```

As primeiras linhas da saída devem ser:

```text
Total de triplas explícitas carregadas do arquivo .ttl: 325
Total de triplas após fechamento dedutivo (OWLRL_Semantics): 1017
Novas triplas inferidas automaticamente pelo Reasoner: 692
Inconsistências lógicas detectadas pelo Reasoner na base: 0
```

In [ ]:
!pip install -q rdflib owlrl
print("Bibliotecas rdflib e owlrl instaladas com sucesso!")

Bibliotecas rdflib e owlrl instaladas com sucesso!


In [ ]:
%%writefile ontologia_doencas.ttl
@prefix :     <http://ufpb.br/ontologia/medicina#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

# =============================================================================
# ONTOLOGIA DE DOENÇAS, SINTOMAS E TRATAMENTOS (Mini-Projeto 3)
# 16 classes | 16 propriedades (9 de objeto + 7 de dados) | 40 instâncias
# Os valores clínicos (letalidade, dosagem, incubação) são ILUSTRATIVOS, para fins
# acadêmicos, e não substituem fontes médicas.
# =============================================================================

# =============================================================================
# T-BOX (TERMINOLOGICAL BOX) — ESQUEMA, HIERARQUIA DE CLASSES E REGRAS OWL
# =============================================================================

# --- Raízes da Hierarquia ---
:EntidadeSaude a owl:Class ;
    rdfs:label "Entidade de Saúde" .

:Pessoa a owl:Class ;
    rdfs:label "Pessoa" .

# --- Nível 1 abaixo de :EntidadeSaude ---
:Doenca a owl:Class ;
    rdfs:subClassOf :EntidadeSaude ;
    owl:disjointWith :Tratamento , :Sintoma ;
    rdfs:label "Doença" .

:Sintoma a owl:Class ;
    rdfs:subClassOf :EntidadeSaude ;
    owl:equivalentClass :ManifestacaoClinica ;
    rdfs:label "Sintoma" .

:ManifestacaoClinica a owl:Class ;
    rdfs:subClassOf :EntidadeSaude ;
    rdfs:label "Manifestação Clínica" .

:Tratamento a owl:Class ;
    rdfs:subClassOf :EntidadeSaude ;
    rdfs:label "Tratamento" .

:AgentePatogenico a owl:Class ;
    rdfs:subClassOf :EntidadeSaude ;
    rdfs:label "Agente Patogênico" .

# --- Nível 2 abaixo de :EntidadeSaude ---
:DoencaInfecciosa a owl:Class ;
    rdfs:subClassOf :Doenca ;
    owl:disjointWith :DoencaCronica ;
    rdfs:label "Doença Infecciosa" .

:DoencaCronica a owl:Class ;
    rdfs:subClassOf :Doenca ;
    rdfs:label "Doença Crônica" .

:Medicamento a owl:Class ;
    rdfs:subClassOf :Tratamento ;
    owl:disjointWith :TerapiaSuporte ;
    rdfs:label "Medicamento" .

:TerapiaSuporte a owl:Class ;
    rdfs:subClassOf :Tratamento ;
    rdfs:label "Terapia de Suporte" .

# --- Nível 3 abaixo de :EntidadeSaude ---
:DoencaViral a owl:Class ;
    rdfs:subClassOf :DoencaInfecciosa ;
    owl:disjointWith :DoencaBacteriana ;
    rdfs:label "Doença Viral" .

:DoencaBacteriana a owl:Class ;
    rdfs:subClassOf :DoencaInfecciosa ;
    rdfs:label "Doença Bacteriana" .

# --- Níveis 1 e 2 abaixo de :Pessoa ---
:Paciente a owl:Class ;
    rdfs:subClassOf :Pessoa ;
    rdfs:label "Paciente" .

:Medico a owl:Class ;
    rdfs:subClassOf :Pessoa ;
    rdfs:label "Médico" .

:PacienteTriagemCritica a owl:Class ;
    rdfs:subClassOf :Paciente ;
    rdfs:label "Paciente em Triagem Crítica" .

# =============================================================================
# PROPRIEDADES DE OBJETO (9)
# =============================================================================

# 1 e 2. Relação Doença <-> Sintoma (Construção OWL: owl:inverseOf)
:apresentaSintoma a owl:ObjectProperty ;
    rdfs:domain :Doenca ;
    rdfs:range :Sintoma ;
    owl:inverseOf :sintomaDe ;
    rdfs:label "apresenta sintoma" .

:sintomaDe a owl:ObjectProperty ;
    rdfs:domain :Sintoma ;
    rdfs:range :Doenca ;
    owl:inverseOf :apresentaSintoma ;
    rdfs:label "é sintoma de" .

# 3 e 4. Relação Doença <-> Tratamento (Construção OWL: owl:inverseOf)
:tratadaCom a owl:ObjectProperty ;
    rdfs:domain :Doenca ;
    rdfs:range :Tratamento ;
    owl:inverseOf :indicadoPara ;
    rdfs:label "tratada com" .

:indicadoPara a owl:ObjectProperty ;
    rdfs:domain :Tratamento ;
    rdfs:range :Doenca ;
    owl:inverseOf :tratadaCom ;
    rdfs:label "indicado para" .

# 5. Doença Infecciosa -> Agente Patogênico PRIMÁRIO (Construção OWL: owl:FunctionalProperty)
#    Modelagem: cada doença infecciosa da base tem UM agente etiológico principal.
#    Se dois agentes forem ligados à mesma doença, o reasoner infere que são o mesmo indivíduo (owl:sameAs).
:causadaPor a owl:ObjectProperty , owl:FunctionalProperty ;
    rdfs:domain :DoencaInfecciosa ;
    rdfs:range :AgentePatogenico ;
    rdfs:label "causada por (agente primário)" .

# 6. Progressão clínica entre doenças (Construção OWL: owl:TransitiveProperty)
:evoluiPara a owl:ObjectProperty , owl:TransitiveProperty ;
    rdfs:domain :Doenca ;
    rdfs:range :Doenca ;
    rdfs:label "pode evoluir para" .

# 7. Interação medicamentosa (Construção OWL: owl:SymmetricProperty)
:interageCom a owl:ObjectProperty , owl:SymmetricProperty ;
    rdfs:domain :Medicamento ;
    rdfs:range :Medicamento ;
    rdfs:label "possui interação medicamentosa com" .

# 8. Diagnóstico do paciente
:diagnosticadoCom a owl:ObjectProperty ;
    rdfs:domain :Paciente ;
    rdfs:range :Doenca ;
    rdfs:label "diagnosticado com" .

# 9. Médico responsável pelo paciente
:atendidoPor a owl:ObjectProperty ;
    rdfs:domain :Paciente ;
    rdfs:range :Medico ;
    rdfs:label "atendido por" .

# =============================================================================
# PROPRIEDADES DE DADOS (7)
# =============================================================================

# 1. Código CID-10 da doença (um único código por doença: owl:FunctionalProperty)
:codigoCID a owl:DatatypeProperty , owl:FunctionalProperty ;
    rdfs:domain :Doenca ;
    rdfs:range xsd:string ;
    rdfs:label "código CID-10" .

# 2. Taxa de letalidade média (%) — valor ilustrativo
:taxaLetalidade a owl:DatatypeProperty ;
    rdfs:domain :Doenca ;
    rdfs:range xsd:decimal ;
    rdfs:label "taxa de letalidade (%)" .

# 3. Período médio de incubação em dias
:periodoIncubacaoDias a owl:DatatypeProperty ;
    rdfs:domain :DoencaInfecciosa ;
    rdfs:range xsd:integer ;
    rdfs:label "período de incubação (dias)" .

# 4. Nível de gravidade do sintoma
:gravidadeSintoma a owl:DatatypeProperty ;
    rdfs:domain :Sintoma ;
    rdfs:range xsd:string ;
    rdfs:label "gravidade do sintoma" .

# 5. Dosagem padrão em miligramas
:dosagemMg a owl:DatatypeProperty ;
    rdfs:domain :Medicamento ;
    rdfs:range xsd:integer ;
    rdfs:label "dosagem padrão (mg)" .

# 6. Idade do paciente (uma única idade por paciente: owl:FunctionalProperty)
:idadePaciente a owl:DatatypeProperty , owl:FunctionalProperty ;
    rdfs:domain :Paciente ;
    rdfs:range xsd:integer ;
    rdfs:label "idade do paciente" .

# 7. Status clínico do paciente no fluxo hospitalar
:statusClinico a owl:DatatypeProperty ;
    rdfs:domain :Paciente ;
    rdfs:range xsd:string ;
    rdfs:label "status clínico" .


# =============================================================================
# A-BOX (ASSERTIONAL BOX) — 40 INSTÂNCIAS DISTRIBUÍDAS ENTRE AS CLASSES
# =============================================================================

# --- 1. Sintomas / Manifestações Clínicas (8 instâncias) ---
:febreAlta a :Sintoma ;
    rdfs:label "Febre Alta" ;
    :gravidadeSintoma "Moderada" .

:cefaleia a :Sintoma ;
    rdfs:label "Cefaleia (Dor de Cabeça)" ;
    :gravidadeSintoma "Leve" .

# :mialgia é declarada como :ManifestacaoClinica (sinônimo de :Sintoma por owl:equivalentClass).
:mialgia a :ManifestacaoClinica ;
    rdfs:label "Mialgia (Dor Muscular)" ;
    :gravidadeSintoma "Moderada" .

# :anosmia é declarada SÓ como :ManifestacaoClinica e não é ligada a nenhuma doença por
# :apresentaSintoma. Logo, a única forma de o reasoner classificá-la como :Sintoma é a
# equivalência de classes (o rdfs:range de :apresentaSintoma não participa).
:anosmia a :ManifestacaoClinica ;
    rdfs:label "Anosmia (Perda de Olfato)" ;
    :gravidadeSintoma "Leve" .

:dispneia a :Sintoma ;
    rdfs:label "Dispneia (Falta de Ar)" ;
    :gravidadeSintoma "Alta" .

:tosseSeca a :Sintoma ;
    rdfs:label "Tosse Seca Persistente" ;
    :gravidadeSintoma "Leve" .

:hemorragia a :Sintoma ;
    rdfs:label "Hemorragia Mucosa" ;
    :gravidadeSintoma "Crítica" .

:rigidezNuca a :Sintoma ;
    rdfs:label "Rigidez na Nuca" ;
    :gravidadeSintoma "Crítica" .

# --- 2. Agentes Patogênicos (6 instâncias) ---
:virusDengue a :AgentePatogenico ;
    rdfs:label "Vírus da Dengue (DENV)" .

:virusInfluenza a :AgentePatogenico ;
    rdfs:label "Vírus Influenza A (H1N1)" .

:sarsCov2 a :AgentePatogenico ;
    rdfs:label "Coronavírus SARS-CoV-2" .

:mycobacteriumTuberculosis a :AgentePatogenico ;
    rdfs:label "Bacilo de Koch (Mycobacterium tuberculosis)" .

:neisseriaMeningitidis a :AgentePatogenico ;
    rdfs:label "Meningococo (Neisseria meningitidis)" .

:streptococcusPneumoniae a :AgentePatogenico ;
    rdfs:label "Pneumococo (Streptococcus pneumoniae)" .

# --- 3. Medicamentos (8) e Terapias de Suporte (2) ---
:paracetamol a :Medicamento ;
    rdfs:label "Paracetamol" ;
    :dosagemMg 750 ;
    :interageCom :rifampicina .   # owl:SymmetricProperty inferirá o sentido inverso

:dipirona a :Medicamento ;
    rdfs:label "Dipirona Sódica" ;
    :dosagemMg 500 .

:oseltamivir a :Medicamento ;
    rdfs:label "Oseltamivir (Tamiflu)" ;
    :dosagemMg 75 .

:amoxicilina a :Medicamento ;
    rdfs:label "Amoxicilina" ;
    :dosagemMg 500 .

:rifampicina a :Medicamento ;
    rdfs:label "Rifampicina" ;
    :dosagemMg 600 .

:ceftriaxona a :Medicamento ;
    rdfs:label "Ceftriaxona" ;
    :dosagemMg 1000 .

:losartana a :Medicamento ;
    rdfs:label "Losartana Potássica" ;
    :dosagemMg 50 ;
    :interageCom :ibuprofeno .    # owl:SymmetricProperty inferirá o sentido inverso

:ibuprofeno a :Medicamento ;
    rdfs:label "Ibuprofeno" ;
    :dosagemMg 400 .

:hidratacaoVenosa a :TerapiaSuporte ;
    rdfs:label "Hidratação Venosa Intensiva" .

:oxigenoterapia a :TerapiaSuporte ;
    rdfs:label "Oxigenoterapia por Cateter Nasal" .

# --- 4. Doenças Virais (5 instâncias) ---
# Obs.: no SUS, A90 cobre as formas de dengue sem manifestação hemorrágica e A91 a forma
# hemorrágica/grave; por isso a dengue com sinais de alarme usa A90.
:dengueClassica a :DoencaViral ;
    rdfs:label "Dengue Clássica" ;
    :codigoCID "A90" ;
    :taxaLetalidade 0.5 ;
    :periodoIncubacaoDias 6 ;
    :causadaPor :virusDengue ;
    :apresentaSintoma :febreAlta , :cefaleia , :mialgia ;
    :tratadaCom :paracetamol , :dipirona , :hidratacaoVenosa ;
    :evoluiPara :dengueAlarme .

:dengueAlarme a :DoencaViral ;
    rdfs:label "Dengue com Sinais de Alarme" ;
    :codigoCID "A90" ;
    :taxaLetalidade 2.5 ;
    :periodoIncubacaoDias 6 ;
    :causadaPor :virusDengue ;
    :apresentaSintoma :febreAlta , :cefaleia , :mialgia , :hemorragia ;
    :tratadaCom :hidratacaoVenosa , :dipirona ;
    :evoluiPara :dengueGrave .

:dengueGrave a :DoencaViral ;
    rdfs:label "Dengue Grave (Hemorrágica)" ;
    :codigoCID "A91" ;
    :taxaLetalidade 12.0 ;
    :periodoIncubacaoDias 6 ;
    :causadaPor :virusDengue ;
    :apresentaSintoma :febreAlta , :hemorragia , :dispneia ;
    :tratadaCom :hidratacaoVenosa , :oxigenoterapia .

:influenzaA a :DoencaViral ;
    rdfs:label "Gripe Influenza A" ;
    :codigoCID "J09" ;
    :taxaLetalidade 1.2 ;
    :periodoIncubacaoDias 3 ;
    :causadaPor :virusInfluenza ;
    :apresentaSintoma :febreAlta , :tosseSeca , :mialgia , :cefaleia ;
    :tratadaCom :oseltamivir , :paracetamol .

:covid19 a :DoencaViral ;
    rdfs:label "COVID-19" ;
    :codigoCID "U07.1" ;
    :taxaLetalidade 2.8 ;
    :periodoIncubacaoDias 5 ;
    :causadaPor :sarsCov2 ;
    :apresentaSintoma :febreAlta , :tosseSeca , :dispneia , :cefaleia ;
    :tratadaCom :dipirona , :oxigenoterapia .

# --- 5. Doenças Bacterianas (3 instâncias) ---
:tuberculose a :DoencaBacteriana ;
    rdfs:label "Tuberculose Pulmonar" ;
    :codigoCID "A15" ;
    :taxaLetalidade 7.5 ;
    :periodoIncubacaoDias 30 ;
    :causadaPor :mycobacteriumTuberculosis ;
    :apresentaSintoma :tosseSeca , :febreAlta , :dispneia ;
    :tratadaCom :rifampicina .

:pneumoniaBacteriana a :DoencaBacteriana ;
    rdfs:label "Pneumonia Bacteriana Aguda" ;
    :codigoCID "J15" ;
    :taxaLetalidade 5.4 ;
    :periodoIncubacaoDias 4 ;
    :causadaPor :streptococcusPneumoniae ;
    :apresentaSintoma :febreAlta , :dispneia , :tosseSeca ;
    :tratadaCom :amoxicilina , :oxigenoterapia .

:meningiteBacteriana a :DoencaBacteriana ;
    rdfs:label "Meningite Meningocócica" ;
    :codigoCID "A39.0" ;
    :taxaLetalidade 18.5 ;
    :periodoIncubacaoDias 4 ;
    :causadaPor :neisseriaMeningitidis ;
    :apresentaSintoma :febreAlta , :cefaleia , :rigidezNuca ;
    :tratadaCom :ceftriaxona , :hidratacaoVenosa .

# --- 6. Doenças Crônicas (2 instâncias) ---
# Para doenças crônicas, a letalidade é ilustrativa (associada a complicações agudas).
:hipertensao a :DoencaCronica ;
    rdfs:label "Hipertensão Arterial Sistêmica" ;
    :codigoCID "I10" ;
    :taxaLetalidade 0.4 ;
    :apresentaSintoma :cefaleia ;
    :tratadaCom :losartana .

:asmaGrave a :DoencaCronica ;
    rdfs:label "Asma Brônquica Grave" ;
    :codigoCID "J45" ;
    :taxaLetalidade 0.8 ;
    :apresentaSintoma :dispneia , :tosseSeca ;
    :tratadaCom :oxigenoterapia .

# --- 7. Médicos (2) e Pacientes (4) ---
:med_helena a :Medico ;
    rdfs:label "Dra. Helena Vasconcelos (Infectologista)" .

:med_carlos a :Medico ;
    rdfs:label "Dr. Carlos Eduardo (Clínico Geral)" .

:pac_lucas a :Paciente ;
    rdfs:label "Lucas Andrade" ;
    :idadePaciente 24 ;
    :statusClinico "Em Triagem" ;
    :diagnosticadoCom :dengueClassica ;
    :atendidoPor :med_helena .

:pac_mariana a :Paciente ;
    rdfs:label "Mariana Costa" ;
    :idadePaciente 68 ;
    :statusClinico "Em Triagem" ;
    :diagnosticadoCom :meningiteBacteriana ;
    :atendidoPor :med_helena .

:pac_roberto a :Paciente ;
    rdfs:label "Roberto Almeida" ;
    :idadePaciente 55 ;
    :statusClinico "Em Triagem" ;
    :diagnosticadoCom :tuberculose ;
    :atendidoPor :med_carlos .

:pac_clara a :Paciente ;
    rdfs:label "Clara Nogueira" ;
    :idadePaciente 31 ;
    :statusClinico "Em Observação" ;
    :diagnosticadoCom :hipertensao ;
    :atendidoPor :med_carlos .

Writing ontologia_doencas.ttl


In [ ]:
%%writefile consultas_doencas.py
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD
from owlrl import DeductiveClosure, OWLRL_Semantics

ARQUIVO_TTL = "ontologia_doencas.ttl"
MED = Namespace("http://ufpb.br/ontologia/medicina#")
# Vocabulário que o owlrl usa para registrar inconsistências lógicas no grafo expandido
ERR = Namespace("http://www.daml.org/2002/03/agents/agent-ont#")


def separar_secao(titulo):
    print("\n" + "=" * 85)
    print(f" {titulo}")
    print("=" * 85)


def nome_curto(uri, g):
    """Retorna o rdfs:label se existir ou o fragmento final da URI."""
    label = g.value(uri, RDFS.label)
    if label:
        return f"{label} (:{str(uri).split('#')[-1]})"
    return str(uri).split("#")[-1] if "#" in str(uri) else str(uri)


def ordenado(triplas):
    """Ordena as triplas para que a saída seja sempre a mesma (reprodutível) a cada execução."""
    return sorted(triplas, key=lambda t: tuple(str(x) for x in t))


def carregar_e_inferir(triplas_extras=()):
    """Carrega o .ttl, acrescenta triplas de teste (opcional) e aplica o reasoner OWL-RL."""
    g = Graph()
    g.parse(ARQUIVO_TTL, format="turtle")
    for t in triplas_extras:
        g.add(t)
    DeductiveClosure(OWLRL_Semantics).expand(g)
    return g


def contar_inconsistencias(g):
    """O owlrl registra cada inconsistência lógica como um nó do tipo err:ErrorMessage."""
    return len(list(g.subjects(RDF.type, ERR.ErrorMessage)))


def main():
    # 1. Carregamento da base Turtle (.ttl)
    g = Graph()
    g.bind("", MED)
    g.bind("med", MED)
    g.bind("rdfs", RDFS)
    g.bind("owl", OWL)

    g.parse(ARQUIVO_TTL, format="turtle")
    triplas_antes = len(g)

    separar_secao("CARREGAMENTO DA ONTOLOGIA E INFERÊNCIA (OWL-RL REASONER)")
    print(f"Total de triplas explícitas carregadas do arquivo .ttl: {triplas_antes}")

    # 2. Execução do Reasoner OWL-RL (expande o grafo in-place com inferências RDFS + OWL)
    DeductiveClosure(OWLRL_Semantics).expand(g)
    triplas_depois = len(g)
    print(f"Total de triplas após fechamento dedutivo (OWLRL_Semantics): {triplas_depois}")
    print(f"Novas triplas inferidas automaticamente pelo Reasoner: {triplas_depois - triplas_antes}")
    print(f"Inconsistências lógicas detectadas pelo Reasoner na base: {contar_inconsistencias(g)}")

    # =========================================================================
    # PARTE 1 — CONSULTAS COM g.triples() (8 PADRÕES VARIADOS)
    # Cada consulta traz: o padrão usado, o que busca e o resultado esperado.
    # =========================================================================
    separar_secao("PARTE 1: CONSULTAS COM g.triples() (PADRÕES VARIADOS)")

    # Consulta 1.1 — padrão (sujeito fixo, predicado fixo, None)
    # O que busca: os sintomas ligados diretamente à Dengue Clássica.
    # Resultado esperado: Febre Alta, Cefaleia e Mialgia.
    print("\n[Consulta 1.1 - g.triples] Padrão (Sujeito Fixo, Predicado Fixo, None)")
    print("Propósito: Quais sintomas a Dengue Clássica apresenta?")
    print("Resultado esperado: Febre Alta, Cefaleia e Mialgia.")
    for s, p, o in ordenado(g.triples((MED.dengueClassica, MED.apresentaSintoma, None))):
        print(f"  -> {nome_curto(o, g)}")

    # Consulta 1.2 — padrão (None, predicado fixo, objeto fixo)
    # O que busca: as doenças que têm 'Dispneia' como sintoma.
    # Resultado esperado: 5 doenças — Dengue Grave, COVID-19, Tuberculose, Pneumonia Bacteriana e Asma Grave.
    print("\n[Consulta 1.2 - g.triples] Padrão (None, Predicado Fixo, Objeto Fixo)")
    print("Propósito: Quais doenças têm 'Dispneia' como sintoma?")
    print("Resultado esperado: Dengue Grave, COVID-19, Tuberculose, Pneumonia Bacteriana e Asma Grave.")
    for s, p, o in ordenado(g.triples((None, MED.apresentaSintoma, MED.dispneia))):
        print(f"  -> {nome_curto(s, g)}")

    # Consulta 1.3 — padrão (sujeito fixo, predicado INFERIDO, None)
    # O que busca: de quais doenças 'Febre Alta' é sintoma, usando :sintomaDe, que NUNCA foi declarada
    # no .ttl (o reasoner a materializa por owl:inverseOf a partir de :apresentaSintoma).
    # Resultado esperado: as 8 doenças infecciosas (3 dengues, Influenza, COVID-19, Tuberculose, Pneumonia e Meningite).
    print("\n[Consulta 1.3 - g.triples] Padrão (Sujeito Fixo, Predicado Inferido owl:inverseOf, None)")
    print("Propósito: 'Febre Alta' é sintoma de quais doenças? (Inferido via owl:inverseOf)")
    print("Resultado esperado: as 8 doenças infecciosas da base.")
    for s, p, o in ordenado(g.triples((MED.febreAlta, MED.sintomaDe, None))):
        print(f"  -> {nome_curto(o, g)}")

    # Consulta 1.4 — padrão (None, predicado transitivo, None)
    # O que busca: todas as relações de evolução clínica, incluindo a deduzida por owl:TransitiveProperty.
    # Resultado esperado: 3 pares — Clássica => Alarme, Alarme => Grave e (inferido) Clássica => Grave.
    print("\n[Consulta 1.4 - g.triples] Padrão (None, Predicado Transitivo :evoluiPara, None)")
    print("Propósito: Quais doenças evoluem para outras (incluindo inferência por transitividade OWL)?")
    print("Resultado esperado: 3 pares, incluindo o salto Dengue Clássica ==> Dengue Grave (não declarado).")
    for s, p, o in ordenado(g.triples((None, MED.evoluiPara, None))):
        print(f"  -> {nome_curto(s, g)}  ==>  {nome_curto(o, g)}")

    # Consulta 1.5 — padrão (sujeito fixo, None, None)
    # O que busca: tudo que se sabe sobre a Rifampicina (filtrando só os predicados de interesse).
    # Resultado esperado: tipos (Medicamento e superclasses), rótulo, dosagem 600, indicadoPara Tuberculose
    # (inverso inferido) e interageCom Paracetamol (simétrico inferido, pois só o Paracetamol o declarou).
    print("\n[Consulta 1.5 - g.triples] Padrão (Sujeito Fixo :rifampicina, None, None)")
    print("Propósito: Todas as afirmações conhecidas/inferidas sobre a Rifampicina:")
    print("Resultado esperado: tipos hierárquicos, rótulo, dosagem 600 mg, indicação para Tuberculose e interação com Paracetamol.")
    for s, p, o in ordenado(g.triples((MED.rifampicina, None, None))):
        pred_nome = str(p).split("#")[-1] if "#" in str(p) else str(p).split("/")[-1]
        if pred_nome in ["type", "label", "dosagemMg", "interageCom", "indicadoPara"]:
            obj_str = nome_curto(o, g) if isinstance(o, URIRef) else str(o)
            print(f"  -> [{pred_nome}] : {obj_str}")

    # Consulta 1.6 — padrão (None, rdf:type, superclasse)
    # O que busca: todas as instâncias de :Doenca, mesmo as declaradas só em subclasses de 3º nível.
    # Resultado esperado: as 10 doenças da base (5 virais, 3 bacterianas e 2 crônicas).
    print("\n[Consulta 1.6 - g.triples] Padrão (None, rdf:type, :Doenca) — Inferência rdfs:subClassOf")
    print("Propósito: Todas as instâncias classificadas como :Doenca pela hierarquia:")
    print("Resultado esperado: as 10 doenças da base.")
    for s, p, o in ordenado(g.triples((None, RDF.type, MED.Doenca))):
        if (s, RDFS.label, None) in g and s != MED.Doenca:
            print(f"  -> {nome_curto(s, g)}")

    # Consulta 1.7 — padrão (None, rdf:type, classe equivalente)
    # O que busca: as instâncias de :Sintoma. Mialgia e Anosmia foram declaradas só como :ManifestacaoClinica;
    # só aparecem aqui por causa de owl:equivalentClass. Anosmia nem participa de :apresentaSintoma.
    # Resultado esperado: os 8 sintomas, incluindo Mialgia e Anosmia.
    print("\n[Consulta 1.7 - g.triples] Padrão (None, rdf:type, :Sintoma) — Inferência owl:equivalentClass")
    print("Propósito: Quais instâncias são :Sintoma (inclusive as declaradas só como :ManifestacaoClinica)?")
    print("Resultado esperado: 8 sintomas, incluindo Mialgia e Anosmia.")
    for s, p, o in ordenado(g.triples((None, RDF.type, MED.Sintoma))):
        if (s, RDFS.label, None) in g and s not in (MED.Sintoma, MED.ManifestacaoClinica):
            print(f"  -> {nome_curto(s, g)}")

    # Consulta 1.8 — padrão (None, None, objeto fixo)
    # O que busca: quem aponta para o Vírus da Dengue, e por qual propriedade (ignora o owl:sameAs reflexivo
    # que o reasoner adiciona a todo indivíduo).
    # Resultado esperado: as 3 formas de dengue, todas via :causadaPor.
    print("\n[Consulta 1.8 - g.triples] Padrão (None, None, Objeto Fixo :virusDengue)")
    print("Propósito: Quais recursos apontam para o Vírus da Dengue e por qual propriedade?")
    print("Resultado esperado: Dengue Clássica, Dengue com Sinais de Alarme e Dengue Grave, via :causadaPor.")
    for s, p, o in ordenado(g.triples((None, None, MED.virusDengue))):
        if p != OWL.sameAs:
            print(f"  -> {nome_curto(s, g)}  --[:{str(p).split('#')[-1]}]-->  {nome_curto(o, g)}")

    # =========================================================================
    # PARTE 2 — CONSULTAS E ATUALIZAÇÕES SPARQL (9 OPERAÇÕES)
    # =========================================================================
    separar_secao("PARTE 2: CONSULTAS DECLARATIVAS E ATUALIZAÇÕES COM SPARQL")

    # -------------------------------------------------------------------------
    # SPARQL 2.1: SELECT + FILTER + ORDER BY
    # Propósito: doenças com letalidade > 2.0%, da mais letal para a menos letal.
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.1 - SELECT + FILTER + ORDER BY]")
    print("Propósito: Doenças com taxa de letalidade > 2.0%, ordenadas de forma decrescente:")
    q1 = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?nomeDoenca ?cid ?letalidade WHERE {
        ?d a :Doenca ;
           rdfs:label ?nomeDoenca ;
           :codigoCID ?cid ;
           :taxaLetalidade ?letalidade .
        FILTER (?letalidade > 2.0)
    }
    ORDER BY DESC(?letalidade)
    """
    for row in g.query(q1):
        print(f"  -> {row.nomeDoenca} (CID: {row.cid}) | Letalidade: {row.letalidade}%")

    # -------------------------------------------------------------------------
    # SPARQL 2.2: SELECT + VALUES + COUNT + GROUP BY + HAVING + ORDER BY (triagem estilo MYCIN)
    # Propósito: o paciente relata Febre Alta, Cefaleia e Mialgia. Ranqueia as doenças candidatas por:
    #   (1) nº de sintomas relatados que a doença apresenta; (2) menor nº total de sintomas da doença
    #   (maior cobertura do quadro); (3) em empate, maior letalidade (conduta de triagem conservadora:
    #   investigar primeiro a hipótese mais perigosa).
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.2 - SELECT + VALUES + COUNT + GROUP BY + HAVING + ORDER BY (Triagem Estilo MYCIN)]")
    print("Propósito: Paciente relata Febre Alta, Cefaleia e Mialgia. Quais as hipóteses diagnósticas mais prováveis?")
    q2 = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?nomeDoenca ?letalidade
           (COUNT(DISTINCT ?sintomaPaciente) AS ?casados)
           (COUNT(DISTINCT ?sintomaDoenca) AS ?totalDoenca)
    WHERE {
        VALUES ?sintomaPaciente { :febreAlta :cefaleia :mialgia }
        ?doenca a :Doenca ;
                rdfs:label ?nomeDoenca ;
                :taxaLetalidade ?letalidade ;
                :apresentaSintoma ?sintomaPaciente ;
                :apresentaSintoma ?sintomaDoenca .
    }
    GROUP BY ?doenca ?nomeDoenca ?letalidade
    HAVING (COUNT(DISTINCT ?sintomaPaciente) >= 2)
    ORDER BY DESC(?casados) ASC(?totalDoenca) DESC(?letalidade)
    """
    for row in g.query(q2):
        cobertura = round(int(row.casados) / int(row.totalDoenca) * 100)
        print(f"  -> Hipótese: {row.nomeDoenca} | {row.casados} de 3 sintomas relatados | "
              f"cobre {cobertura}% do quadro da doença | letalidade {row.letalidade}%")

    # -------------------------------------------------------------------------
    # SPARQL 2.3: SELECT + OPTIONAL + BIND
    # Propósito: listar doenças, agente patogênico (opcional: doenças crônicas não têm) e risco via IF.
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.3 - SELECT + OPTIONAL + BIND]")
    print("Propósito: Listar doenças, agente etiológico (opcional) e classificação automática de risco:")
    q3 = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?nomeDoenca ?nomeAgente ?letalidade ?nivelRisco WHERE {
        ?d a :Doenca ;
           rdfs:label ?nomeDoenca ;
           :taxaLetalidade ?letalidade .
        OPTIONAL {
            ?d :causadaPor ?ag .
            ?ag rdfs:label ?nomeAgente .
        }
        BIND(IF(?letalidade >= 5.0, "ALTO RISCO", "RISCO MODERADO/BAIXO") AS ?nivelRisco)
    }
    ORDER BY DESC(?letalidade)
    """
    for row in g.query(q3):
        agente = row.nomeAgente if row.nomeAgente else "Sem agente infeccioso registrado (doença crônica)"
        print(f"  -> {row.nomeDoenca} | Patógeno: {agente} | [{row.nivelRisco}]")

    # -------------------------------------------------------------------------
    # SPARQL 2.4: SELECT com junções múltiplas + GROUP_CONCAT
    # Propósito: prontuário integrado Paciente -> Doença -> Tratamentos, com o médico responsável.
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.4 - SELECT com Junções Múltiplas]")
    print("Propósito: Prontuário integrado (Paciente, Idade, Doença Diagnosticada, Médico e Tratamentos):")
    q4 = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?nomePaciente ?idade ?nomeDoenca ?nomeMedico (GROUP_CONCAT(?nomeTrat; separator=", ") AS ?tratamentos)
    WHERE {
        ?p a :Paciente ;
           rdfs:label ?nomePaciente ;
           :idadePaciente ?idade ;
           :diagnosticadoCom ?d ;
           :atendidoPor ?m .
        ?d rdfs:label ?nomeDoenca ;
           :tratadaCom ?t .
        ?t rdfs:label ?nomeTrat .
        ?m rdfs:label ?nomeMedico .
    }
    GROUP BY ?nomePaciente ?idade ?nomeDoenca ?nomeMedico
    ORDER BY DESC(?idade)
    """
    for row in g.query(q4):
        print(f"  -> Paciente: {row.nomePaciente} ({row.idade} anos) | Doença: {row.nomeDoenca}")
        print(f"     Médico: {row.nomeMedico} | Conduta: {row.tratamentos}")

    # -------------------------------------------------------------------------
    # SPARQL 2.5: ASK
    # Propósito: existe interação entre Ibuprofeno e Losartana? Só a Losartana declarou a interação no .ttl;
    # a consulta pergunta no sentido oposto, então o True vem de owl:SymmetricProperty.
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.5 - ASK (Verificação de Interação Medicamentosa)]")
    q5_ask = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    ASK {
        :ibuprofeno :interageCom :losartana .
    }
    """
    existe_interacao = bool(g.query(q5_ask))
    print(f"Propósito: Existe alerta de interação medicamentosa entre Ibuprofeno e Losartana? -> {existe_interacao}")

    # -------------------------------------------------------------------------
    # SPARQL 2.6: CONSTRUCT
    # Propósito: derivar um novo grafo de prescrições diretas (?paciente :prescricaoSugerida ?tratamento).
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.6 - CONSTRUCT (Derivando Grafo de Prescrições Médicas)]")
    print("Propósito: Construir triplas diretas (?paciente :prescricaoSugerida ?tratamento):")
    q6_construct = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    CONSTRUCT {
        ?pac :prescricaoSugerida ?trat .
    }
    WHERE {
        ?pac a :Paciente ;
             :diagnosticadoCom ?doenca .
        ?doenca :tratadaCom ?trat .
    }
    """
    grafo_prescricoes = g.query(q6_construct).graph
    print(f"Total de triplas geradas no subgrafo CONSTRUCT: {len(grafo_prescricoes)}")
    for s, p, o in ordenado(grafo_prescricoes):
        print(f"  -> {nome_curto(s, g)}  --[:prescricaoSugerida]-->  {nome_curto(o, g)}")

    # -------------------------------------------------------------------------
    # SPARQL 2.7: UPDATE — INSERT...WHERE (regra de produção)
    # Propósito: classificar como :PacienteTriagemCritica todo paciente cuja doença tenha letalidade >= 5.0%.
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.7 - UPDATE: INSERT...WHERE (Regra de Produção para Triagem Crítica)]")
    antes_critico = [nome_curto(s, g) for s, _, _ in ordenado(g.triples((None, RDF.type, MED.PacienteTriagemCritica)))]
    print(f"Pacientes em :PacienteTriagemCritica ANTES do INSERT: {antes_critico if antes_critico else 'nenhum'}")
    q7_insert = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    INSERT {
        ?pac a :PacienteTriagemCritica .
    }
    WHERE {
        ?pac a :Paciente ;
             :diagnosticadoCom ?doenca .
        ?doenca :taxaLetalidade ?let .
        FILTER (?let >= 5.0)
    }
    """
    g.update(q7_insert)
    print("Pacientes em :PacienteTriagemCritica DEPOIS do INSERT:")
    for s, p, o in ordenado(g.triples((None, RDF.type, MED.PacienteTriagemCritica))):
        print(f"  -> {nome_curto(s, g)}")

    # -------------------------------------------------------------------------
    # SPARQL 2.8: UPDATE — DELETE DATA
    # Propósito: revisão de protocolo — o Paracetamol deixa de ser indicado para a Dengue Clássica.
    # Remove também a tripla inversa (:indicadoPara), que o reasoner já havia materializado no grafo.
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.8 - UPDATE: DELETE DATA]")
    print("Propósito: Revisão de protocolo — o Paracetamol deixa de ser indicado para a Dengue Clássica.")
    print("           Removemos também a tripla inversa (:indicadoPara), que o reasoner já havia materializado.")
    trat_antes = [nome_curto(o, g) for _, _, o in ordenado(g.triples((MED.dengueClassica, MED.tratadaCom, None)))]
    ind_antes = [nome_curto(o, g) for _, _, o in ordenado(g.triples((MED.paracetamol, MED.indicadoPara, None)))]
    print(f"Tratamentos da Dengue Clássica ANTES do DELETE: {trat_antes}")
    print(f"Paracetamol é indicado para ANTES do DELETE: {ind_antes}")

    q8_delete = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    DELETE DATA {
        :dengueClassica :tratadaCom :paracetamol .
        :paracetamol :indicadoPara :dengueClassica .
    }
    """
    g.update(q8_delete)
    trat_depois = [nome_curto(o, g) for _, _, o in ordenado(g.triples((MED.dengueClassica, MED.tratadaCom, None)))]
    ind_depois = [nome_curto(o, g) for _, _, o in ordenado(g.triples((MED.paracetamol, MED.indicadoPara, None)))]
    print(f"Tratamentos da Dengue Clássica DEPOIS do DELETE: {trat_depois}")
    print(f"Paracetamol é indicado para DEPOIS do DELETE: {ind_depois}")

    # -------------------------------------------------------------------------
    # SPARQL 2.9: UPDATE — DELETE...INSERT...WHERE (transição de estado)
    # Propósito: pacientes "Em Triagem" que já têm médico atribuído passam a "Em Tratamento Supervisionado".
    # A paciente "Em Observação" não casa com o WHERE e permanece como está.
    # -------------------------------------------------------------------------
    print("\n[SPARQL 2.9 - UPDATE: DELETE...INSERT...WHERE (Transição de Estado Clínico)]")
    print("Status dos pacientes ANTES do DELETE/INSERT:")
    for pac, _, st in ordenado(g.triples((None, MED.statusClinico, None))):
        print(f"  -> {nome_curto(pac, g)}: '{st}'")

    q9_del_ins = """
    PREFIX : <http://ufpb.br/ontologia/medicina#>
    DELETE {
        ?pac :statusClinico "Em Triagem" .
    }
    INSERT {
        ?pac :statusClinico "Em Tratamento Supervisionado" .
    }
    WHERE {
        ?pac a :Paciente ;
             :statusClinico "Em Triagem" ;
             :atendidoPor ?med .
    }
    """
    g.update(q9_del_ins)

    print("Status dos pacientes DEPOIS do DELETE/INSERT:")
    for pac, _, st in ordenado(g.triples((None, MED.statusClinico, None))):
        print(f"  -> {nome_curto(pac, g)}: '{st}'")

    # =========================================================================
    # PARTE 3 — VERIFICAÇÃO DAS CONSTRUÇÕES OWL (grafos de teste isolados)
    # Em OWL, FunctionalProperty e disjointWith NÃO bloqueiam dados (isso seria papel do SHACL):
    # o reasoner INFERE ou DETECTA. Cada teste usa uma cópia nova do .ttl com uma tripla extra,
    # sem alterar o grafo principal acima.
    # =========================================================================
    separar_secao("PARTE 3: VERIFICAÇÃO DAS CONSTRUÇÕES OWL (GRAFOS DE TESTE ISOLADOS)")

    # Verificação 3.1 — owl:FunctionalProperty em :causadaPor
    # Teste: ligar a Dengue Clássica a um SEGUNDO agente (:denvSorotipo1), além de :virusDengue.
    # Resultado esperado: True — como :causadaPor é funcional, os dois objetos só podem ser o mesmo
    # indivíduo, e o reasoner infere (:virusDengue owl:sameAs :denvSorotipo1).
    g1 = carregar_e_inferir([(MED.dengueClassica, MED.causadaPor, MED.denvSorotipo1)])
    mesmo_agente = (MED.virusDengue, OWL.sameAs, MED.denvSorotipo1) in g1
    print("\n[Verificação 3.1 - owl:FunctionalProperty]")
    print("Teste: Dengue Clássica ligada a dois agentes via :causadaPor (:virusDengue e :denvSorotipo1).")
    print("Resultado esperado: True (o reasoner infere que os dois agentes são o mesmo indivíduo).")
    print(f"  -> (:virusDengue owl:sameAs :denvSorotipo1) inferido? {mesmo_agente}")

    # Verificação 3.2 — owl:disjointWith (:DoencaViral x :DoencaBacteriana)
    # Teste: classificar o mesmo indivíduo como viral e bacteriano ao mesmo tempo.
    # Resultado esperado: base original = 0 inconsistências; grafo de teste >= 1.
    g2 = carregar_e_inferir([(MED.infeccaoImpossivel, RDF.type, MED.DoencaViral),
                             (MED.infeccaoImpossivel, RDF.type, MED.DoencaBacteriana)])
    print("\n[Verificação 3.2 - owl:disjointWith]")
    print("Teste: um mesmo indivíduo declarado :DoencaViral e :DoencaBacteriana (classes disjuntas).")
    print("Resultado esperado: 0 inconsistências na base original e 1 ou mais no grafo de teste.")
    print(f"  -> Inconsistências na base original: {contar_inconsistencias(carregar_e_inferir())}")
    print(f"  -> Inconsistências no grafo de teste: {contar_inconsistencias(g2)}")

    # Verificação 3.3 — owl:equivalentClass (:Sintoma = :ManifestacaoClinica)
    # Teste: :anosmia foi declarada só como :ManifestacaoClinica e nenhuma doença a referencia.
    # Resultado esperado: True e True — foi classificada como :Sintoma exclusivamente pela equivalência.
    g3 = carregar_e_inferir()
    anosmia_sintoma = (MED.anosmia, RDF.type, MED.Sintoma) in g3
    anosmia_isolada = (None, MED.apresentaSintoma, MED.anosmia) not in g3
    print("\n[Verificação 3.3 - owl:equivalentClass]")
    print("Teste: :anosmia (declarada só como :ManifestacaoClinica) virou :Sintoma sem passar por :apresentaSintoma?")
    print("Resultado esperado: True e True.")
    print(f"  -> :anosmia é :Sintoma? {anosmia_sintoma} | nenhuma doença a referencia via :apresentaSintoma? {anosmia_isolada}")


if __name__ == "__main__":
    main()

Overwriting consultas_doencas.py


In [ ]:
!python consultas_doencas.py


 CARREGAMENTO DA ONTOLOGIA E INFERÊNCIA (OWL-RL REASONER)
Total de triplas explícitas carregadas do arquivo .ttl: 325
Total de triplas após fechamento dedutivo (OWLRL_Semantics): 1017
Novas triplas inferidas automaticamente pelo Reasoner: 692
Inconsistências lógicas detectadas pelo Reasoner na base: 0

 PARTE 1: CONSULTAS COM g.triples() (PADRÕES VARIADOS)

[Consulta 1.1 - g.triples] Padrão (Sujeito Fixo, Predicado Fixo, None)
Propósito: Quais sintomas a Dengue Clássica apresenta?
Resultado esperado: Febre Alta, Cefaleia e Mialgia.
  -> Cefaleia (Dor de Cabeça) (:cefaleia)
  -> Febre Alta (:febreAlta)
  -> Mialgia (Dor Muscular) (:mialgia)

[Consulta 1.2 - g.triples] Padrão (None, Predicado Fixo, Objeto Fixo)
Propósito: Quais doenças têm 'Dispneia' como sintoma?
Resultado esperado: Dengue Grave, COVID-19, Tuberculose, Pneumonia Bacteriana e Asma Grave.
  -> Asma Brônquica Grave (:asmaGrave)
  -> COVID-19 (:covid19)
  -> Dengue Grave (Hemorrágica) (:dengueGrave)
  -> Pneumonia Bacteria

In [ ]:
# Instala a biblioteca de grafos interativos
!pip install -q pyvis

from pyvis.network import Network
from rdflib import Graph, Literal
import IPython

# 1. Carrega o grafo
g = Graph()
g.parse("ontologia_doencas.ttl", format="turtle")

# 2. Cria a rede interativa (com física aplicada para afastar nós)
net = Network(notebook=True, directed=True, width="100%", height="800px", cdn_resources='remote')
net.force_atlas_2based(spring_length=150) # Algoritmo que organiza os nós de forma orgânica

def extrair_nome(nodo):
    txt = str(nodo)
    if isinstance(nodo, Literal):
        return txt[:25] + "..." if len(txt) > 25 else txt
    return txt.split("#")[-1] if "#" in txt else txt.split("/")[-1]

# 3. Adiciona os nós e as arestas
for s, p, o in g:
    s_nome = extrair_nome(s)
    p_nome = extrair_nome(p)
    o_nome = extrair_nome(o)

    # Adiciona nós garantindo que não duplicam (o PyVis lida com isso se o ID for igual)
    net.add_node(s_nome, title=s_nome, label=s_nome, color="#4CAF50")
    net.add_node(o_nome, title=o_nome, label=o_nome, color="#2196F3")

    # Adiciona a seta conectando
    net.add_edge(s_nome, o_nome, title=p_nome, label=p_nome, color="gray")

# 4. Salva e exibe no Colab
net.show("ontologia_interativa.html")
IPython.display.HTML("ontologia_interativa.html")

ontologia_interativa.html


In [ ]:
!pip install -q python-pptx
import collections
import collections.abc
from pptx import Presentation

prs = Presentation('/content/Mini_Projeto_03_-_Roteiro.pptx')
for i, slide in enumerate(prs.slides):
    print(f"=== Slide {i+1} ===")
    for shape in slide.shapes:
        if hasattr(shape, "text") and shape.text.strip():
            print(shape.text.strip())

=== Slide 1 ===
Mini-Projeto 3 — especificação
Modelar uma base de conhecimento com RDF, RDFS e OWL, entregue em Turtle (.ttl), e consultá-la com rdflib e SPARQL
Requisitos mínimos da base (.ttl):
8+ classes organizadas em hierarquia (rdfs:subClassOf) com pelo menos 2 níveis abaixo da raiz.10+ propriedades (mínimo 5 de objeto e 5 de dados), use quando couber rdfs:domain e rdfs:range declarados.25+ instâncias distribuídas entre as classes, com valores plausíveis no domínio.50+ triplas no total no arquivo Turtle.5+ construções OWL justificadas pelo domínio (ex.: owl:inverseOf, owl:TransitiveProperty, owl:FunctionalProperty).
Requisitos mínimos de consulta:
Parte 1, rdflib: 5+ consultas com g.triples(), variando os padrões (sujeito fixo, predicado fixo, objeto fixo e combinações com None).Parte 2, SPARQL: 8+ consultas sobre a mesma base, cobrindo obrigatoriamente:  SELECT (ao menos um com FILTER e um com ORDER BY ou agregação),  ASK,  CONSTRUCT e  Update: 1 INSERT, 1 DELETE e 1 DELET